# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haideriqbal499/Flyrank_Ml_internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane locked: Lane 2 — Refresh / Content Opportunity Scoring.**

Check two signals, encode one transparent rule (score + one reason code + action),
write the ranked queue, then review the top ten with a skeptic's eye.

> Skills: `building-baselines` + `flyrank/flyrank-data`.

In [1]:
import os
import sys
import json
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/haideriqbal499/Flyrank_Ml_internship"
REPO_DIR = "Flyrank_Ml_internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != os.path.abspath(os.sep):
        os.chdir("..")

CSV = Path("data/raw/content_refresh_anonymized.csv")
OUT_DIR = Path("work/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
QUEUE_PATH = OUT_DIR / "baseline_action_score.csv"
METRICS_PATH = OUT_DIR / "baseline_action_score_metrics.json"

df = pd.read_csv(CSV)
# Snapshot proxy for signal checks / precision@K only — NEVER a rule input.
df["is_declining_label"] = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)
# avg_position == 0 means missing, not rank zero
df["has_position"] = df["avg_position"] > 0

print("Working dir:", os.getcwd())
print(f"Loaded {len(df):,} rows")
print("Lane: 2 — Refresh / Content Opportunity Scoring")
print(f"Declining-label rate: {df['is_declining_label'].mean():.3f}")

Working dir: C:\Users\Windows 11\Flyrank_Ml_internship
Loaded 30,000 rows
Lane: 2 — Refresh / Content Opportunity Scoring
Declining-label rate: 0.542


## 1. Two signal checks + the rule in plain words

I checked two candidate signals before locking the rule. A negative verdict is allowed — it
just means that idea does not earn a place in *this* baseline.

**Final rule (plain words):** Among pages that already have real demand and a usable position
in the top ~20, prioritize those whose CTR is weak for that visibility. That is the same idea
behind FlyRank's **CTR-fix / needs_ctr_fix** logic.

**One reason code:** `low_ctr_visible_page`  
**Action:** `refresh_and_review_ctr`

### Signal A — Staleness (flag-linked: refresh / stale-visible)

Question: do pages with longer `days_since_last_update` show a higher declining-label rate?

In [2]:
staleness = df.copy()
staleness["stale_bucket"] = pd.cut(
    staleness["days_since_last_update"],
    bins=[-np.inf, 90, 180, 365, np.inf],
    labels=["0-90d", "91-180d", "181-365d", "365d+"],
)

sig_a = (
    staleness.groupby("stale_bucket", observed=True)
    .agg(
        n=("is_declining_label", "size"),
        declining_rate=("is_declining_label", "mean"),
        median_impressions=("impressions_90d", "median"),
    )
    .reset_index()
)
sig_a["declining_rate"] = sig_a["declining_rate"].round(3)
print("Signal A — days_since_last_update vs declining-label rate")
print(sig_a.to_string(index=False))
print(f"\nOverall declining-label rate: {df['is_declining_label'].mean():.3f}")

rate_fresh = float(sig_a.loc[sig_a["stale_bucket"] == "0-90d", "declining_rate"].iloc[0])
rate_181 = float(sig_a.loc[sig_a["stale_bucket"] == "181-365d", "declining_rate"].iloc[0])
n_181 = int(sig_a.loc[sig_a["stale_bucket"] == "181-365d", "n"].iloc[0])
rate_mid = float(sig_a.loc[sig_a["stale_bucket"] == "91-180d", "declining_rate"].iloc[0])

# Deep-stale (180d+) is the refresh-flag threshold; judge that bucket, not a tiny 365d+ tail.
if n_181 < 50:
    verdict_a = "MIXED"
    why_a = (
        f"181-365d has only n={n_181}; mid bucket 91-180d rate={rate_mid:.3f}. "
        "Too little deep-stale mass to trust a staleness rule here."
    )
elif rate_181 > rate_fresh + 0.03:
    verdict_a = "CONFIRMED"
    why_a = f"181-365d declining_rate={rate_181:.3f} > fresh {rate_fresh:.3f}"
elif rate_181 < rate_fresh - 0.03:
    verdict_a = "OPPOSITE"
    why_a = (
        f"181-365d declining_rate={rate_181:.3f} < fresh {rate_fresh:.3f} "
        f"(n={n_181}). Staleness-behind-refresh does not hold cleanly on this snapshot — "
        f"saved us from building the baseline on it. Mid bucket 91-180d is hotter ({rate_mid:.3f})."
    )
else:
    verdict_a = "FALSE"
    why_a = f"181-365d ~= fresh ({rate_181:.3f} vs {rate_fresh:.3f})"

print(f"\nVerdict A (staleness): {verdict_a}")
print("Why:", why_a)

Signal A — days_since_last_update vs declining-label rate
stale_bucket     n  declining_rate  median_impressions
       0-90d 20655           0.512               472.0
     91-180d  9171           0.611              1692.0
    181-365d   169           0.467                16.0
       365d+     5           0.600                 2.0

Overall declining-label rate: 0.542

Verdict A (staleness): OPPOSITE
Why: 181-365d declining_rate=0.467 < fresh 0.512 (n=169). Staleness-behind-refresh does not hold cleanly on this snapshot — saved us from building the baseline on it. Mid bucket 91-180d is hotter (0.611).


### Signal B — CTR vs position (flag-linked: needs_ctr_fix / CTR-fix logic)

Question: among visible pages (`impressions_90d >= 500`, position 1–20), do low-CTR pages
show a higher declining-label rate?

In [3]:
visible = df[
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
].copy()

# ctr is a ×100 percentage in this dataset (0.76 means 0.76%, not 76%).
visible["ctr_bucket"] = pd.cut(
    visible["ctr"],
    bins=[-np.inf, 0.5, 1.0, 2.0, np.inf],
    labels=["<0.5", "0.5-1", "1-2", "2+"],
)

sig_b = (
    visible.groupby("ctr_bucket", observed=True)
    .agg(
        n=("is_declining_label", "size"),
        declining_rate=("is_declining_label", "mean"),
        median_position=("avg_position", "median"),
    )
    .reset_index()
)
sig_b["declining_rate"] = sig_b["declining_rate"].round(3)
print("Signal B — CTR buckets among visible pages (imp>=500, pos 1-20)")
print(sig_b.to_string(index=False))
print(f"Visible slice n={len(visible):,}")

rate_low_ctr = float(sig_b.loc[sig_b["ctr_bucket"] == "<0.5", "declining_rate"].iloc[0])
rate_mid_ctr = float(sig_b.loc[sig_b["ctr_bucket"] == "0.5-1", "declining_rate"].iloc[0])
n_low = int(sig_b.loc[sig_b["ctr_bucket"] == "<0.5", "n"].iloc[0])

if rate_low_ctr > rate_mid_ctr + 0.03:
    verdict_b = "CONFIRMED"
elif rate_low_ctr < rate_mid_ctr - 0.03:
    verdict_b = "OPPOSITE"
else:
    verdict_b = "MIXED"

print(f"\nVerdict B (CTR vs position): {verdict_b}")
print(
    f"Why: among visible pages, CTR<0.5 declining_rate={rate_low_ctr:.3f} "
    f"(n={n_low:,}) vs CTR 0.5-1 rate={rate_mid_ctr:.3f}. "
    "This is the signal the baseline will use."
)

Signal B — CTR buckets among visible pages (imp>=500, pos 1-20)
ctr_bucket    n  declining_rate  median_position
      <0.5 9822           0.627              8.4
     0.5-1 1681           0.477              7.1
       1-2  473           0.455              7.5
        2+   47           0.532              9.0
Visible slice n=12,023

Verdict B (CTR vs position): CONFIRMED
Why: among visible pages, CTR<0.5 declining_rate=0.627 (n=9,822) vs CTR 0.5-1 rate=0.477. This is the signal the baseline will use.


## 2. Build the ranked queue (writes the CSV)

**ONE rule:**
- Fire when `impressions_90d >= 500` and `0 < avg_position <= 20` and `ctr < 0.5`.
- `reason_code` = `low_ctr_visible_page` (else `monitor_not_ctr_gap`).
- `action` = `refresh_and_review_ctr` when it fires, else `monitor`.
- `score` = `log1p(impressions_90d) * gate` so higher-demand CTR gaps rank first.

No `trend_direction` / `trend_pct` / `is_declining_label` / product flags in the score.

In [4]:
queue = df.copy()

queue["ctr_gap"] = (
    (queue["impressions_90d"] >= 500)
    & (queue["avg_position"] > 0)
    & (queue["avg_position"] <= 20)
    & (queue["ctr"] < 0.5)
).astype(int)

queue["reason_code"] = np.where(
    queue["ctr_gap"] == 1, "low_ctr_visible_page", "monitor_not_ctr_gap"
)
queue["action"] = np.where(queue["ctr_gap"] == 1, "refresh_and_review_ctr", "monitor")
queue["score"] = np.log1p(queue["impressions_90d"]) * queue["ctr_gap"]

queue = queue.sort_values(["score", "impressions_90d"], ascending=[False, False]).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)

out_cols = [
    "rank",
    "content_id",
    "client_id",
    "score",
    "reason_code",
    "action",
    "impressions_90d",
    "avg_position",
    "ctr",
    "days_since_last_update",
    "is_declining_label",
]
ranked = queue[out_cols].copy()
ranked.to_csv(QUEUE_PATH, index=False)

n_fire = int(queue["ctr_gap"].sum())
base_rate = float(df["is_declining_label"].mean())
top10 = ranked.head(10)
top50 = ranked.head(50)
p_at_10 = float(top10["is_declining_label"].mean())
p_at_50 = float(top50["is_declining_label"].mean())

metrics = {
    "lane": "Lane 2: Refresh / Content Opportunity Scoring",
    "rule": "impressions_90d >= 500 AND 0 < avg_position <= 20 AND ctr < 0.5",
    "reason_code": "low_ctr_visible_page",
    "action": "refresh_and_review_ctr",
    "rows": int(len(ranked)),
    "rows_firing_rule": n_fire,
    "base_rate_declining": round(base_rate, 4),
    "precision_at_10_vs_declining_label": round(p_at_10, 4),
    "precision_at_50_vs_declining_label": round(p_at_50, 4),
    "signal_verdicts": {"staleness": verdict_a, "ctr_vs_position": verdict_b},
    "inputs_used": ["impressions_90d", "avg_position", "ctr"],
    "inputs_excluded": ["trend_direction", "trend_pct", "is_declining_label", "product flags"],
    "queue_path": "work/outputs/baseline_action_score.csv",
}
METRICS_PATH.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

print(f"Wrote {QUEUE_PATH} ({len(ranked):,} rows)")
print(f"Wrote {METRICS_PATH}")
print(f"Rule fired on {n_fire:,} / {len(ranked):,} pages")
print(f"Base rate (declining label): {base_rate:.3f}")
print(f"Precision@10 vs declining label: {p_at_10:.3f}")
print(f"Precision@50 vs declining label: {p_at_50:.3f}")
print("\nTop 10 preview:")
print(
    top10[
        ["rank", "score", "reason_code", "action", "impressions_90d", "avg_position", "ctr", "is_declining_label"]
    ].to_string(index=False)
)

Wrote work\outputs\baseline_action_score.csv (30,000 rows)
Wrote work\outputs\baseline_action_score_metrics.json
Rule fired on 9,759 / 30,000 pages
Base rate (declining label): 0.542
Precision@10 vs declining label: 0.600
Precision@50 vs declining label: 0.420

Top 10 preview:
 rank     score          reason_code                 action  impressions_90d  avg_position  ctr  is_declining_label
    1 13.157182 low_ctr_visible_page refresh_and_review_ctr           517715           4.2 0.14                   1
    2 13.156011 low_ctr_visible_page refresh_and_review_ctr           517109           5.4 0.25                   0
    3 13.140700 low_ctr_visible_page refresh_and_review_ctr           509252           2.5 0.15                   1
    4 13.045707 low_ctr_visible_page refresh_and_review_ctr           463103           2.3 0.41                   1
    5 12.938876 low_ctr_visible_page refresh_and_review_ctr           416180           4.0 0.23                   1
    6 12.751624 low_ctr_vi

## 3. Top-10 review

For each of the top ten: action, why it's there, and what would make it wrong.

In [5]:
reviews = []
for _, row in top10.iterrows():
    why = (
        f"low_ctr_visible_page: pos={row['avg_position']:.1f}, "
        f"ctr={row['ctr']:.3f} (×100 scale), "
        f"impressions_90d={int(row['impressions_90d']):,}, score={row['score']:.2f}"
    )
    wrong = (
        "Wrong if CTR is low because of a non-search landing intent, a soft SERP feature "
        "stealing clicks, a branded query with inevitable low CTR, or the title/meta was "
        "already fixed after this snapshot."
    )
    reviews.append(
        {
            "rank": int(row["rank"]),
            "content_id": row["content_id"],
            "action": row["action"],
            "why": why,
            "what_would_make_it_wrong": wrong,
            "declining_label": int(row["is_declining_label"]),
        }
    )

review_df = pd.DataFrame(reviews)
print(review_df[["rank", "content_id", "action", "declining_label"]].to_string(index=False))
print("\n--- Line-by-line skeptic notes ---")
for r in reviews:
    print(f"#{r['rank']} {r['content_id']}")
    print(f"  action: {r['action']}")
    print(f"  why: {r['why']}")
    print(f"  wrong if: {r['what_would_make_it_wrong']}")

 rank           content_id                 action  declining_label
    1 content_5fe46e04994d refresh_and_review_ctr                1
    2 content_aaef01a50def refresh_and_review_ctr                0
    3 content_8c19996aa890 refresh_and_review_ctr                1
    4 content_4c36c775b818 refresh_and_review_ctr                1
    5 content_1a9e894be2e2 refresh_and_review_ctr                1
    6 content_db5989a78dd3 refresh_and_review_ctr                0
    7 content_cb112fce36be refresh_and_review_ctr                1
    8 content_36ff89c8214e refresh_and_review_ctr                0
    9 content_8451fc6f034d refresh_and_review_ctr                0
   10 content_008fb02c46cb refresh_and_review_ctr                1

--- Line-by-line skeptic notes ---
#1 content_5fe46e04994d
  action: refresh_and_review_ctr
  why: low_ctr_visible_page: pos=4.2, ctr=0.140 (x100 scale), impressions_90d=517,715, score=13.16
  wrong if: Wrong if CTR is low because of a non-search landing intent,

## 4. Weak picks + leakage check

**What the signals decided:** Signal A (staleness) did **not** earn the baseline; Signal B
(CTR vs position) did. That is the point of checking first.

**Leakage:** score inputs are only `impressions_90d`, `avg_position`, and `ctr`.
`trend_*` / `is_declining_label` never enter the score.

In [6]:
print("Score inputs:", ["impressions_90d", "avg_position", "ctr"])
print("Excluded from score:", ["trend_direction", "trend_pct", "is_declining_label", "product flags"])
print("Max score among non-firing rows (must be 0):", float(queue.loc[queue["ctr_gap"] == 0, "score"].max()))

weak = top10.loc[top10["is_declining_label"] == 0]
print("\nTop-10 with declining_label=0 (skeptic: still may be fair CTR reviews):")
if len(weak):
    print(weak[["rank", "content_id", "impressions_90d", "avg_position", "ctr"]].to_string(index=False))
else:
    print("(none in this run — look for branded/SERP-feature false friends in a later hand pass)")

print("\nLane lock confirmation: Lane 2 stays.")
print("Signal verdicts:", {"staleness": verdict_a, "ctr_vs_position": verdict_b})

Score inputs: ['impressions_90d', 'avg_position', 'ctr']
Excluded from score: ['trend_direction', 'trend_pct', 'is_declining_label', 'product flags']
Max score among non-firing rows (must be 0): 0.0

Top-10 with declining_label=0 (skeptic: still may be fair CTR reviews):
 rank           content_id  impressions_90d  avg_position  ctr
    2 content_aaef01a50def           517109           5.4 0.25
    6 content_db5989a78dd3           345111           5.4 0.21
    8 content_36ff89c8214e           295097           7.3 0.05
    9 content_8451fc6f034d           272144           2.3 0.03

Lane lock confirmation: Lane 2 stays.
Signal verdicts: {'staleness': 'OPPOSITE', 'ctr_vs_position': 'CONFIRMED'}


## Self-check

- [x] Two signal verdicts with bucket tables and **n** (staleness flag-linked; CTR-vs-position flag-linked)
- [x] One rule: score + one reason code (`low_ctr_visible_page`) + action (`refresh_and_review_ctr`)
- [x] Ranked queue written to `work/outputs/baseline_action_score.csv`
- [x] Top-10 reviewed with action / why / what would make it wrong
- [x] No future-window or label-derived inputs in the score
- [x] Lane confirmed: **Lane 2**
- [x] Notebook executed top to bottom (outputs visible)
- [x] Committed under `work/notebooks/` (+ metrics JSON); CSV stays gitignored